In [1]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.concepts import vehicle_knowledge_graph

import prism


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [3]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [4]:
prep_data.keys()

dict_keys(['battery_materials', 'battery_shares', 'weight_boats', 'material_fractions', 'battery_weights', 'stocks', 'shares', 'lifetimes', 'weights'])

In [4]:
import pandas as pd
from imagematerials.util import dataset_to_array

# Copy dimensiomns from material_fractions for xr_maintenance_material
materials = prep_data['material_fractions'].coords["material"]
types = prep_data['material_fractions'].coords["Type"]

maintenance_material_pd : pd.DataFrame = pd.read_csv(
    "../data/raw/vehicles/standard_data/all_vehicle_maintenance_image.csv", index_col=0)

maintenance_material_pd['Li'] = 0
maintenance_material_pd['Mn'] = 0
maintenance_material_pd['Ni'] = 0
maintenance_material_pd['Ti'] = 0

stacked_maintenance_material = maintenance_material_pd.set_index("Type").stack().rename_axis(index=["Type", "material"]).reset_index(name="value")

stacked_maintenance_material = stacked_maintenance_material.set_index(["Type", "material"])

stacked_maintenance_material_xr = stacked_maintenance_material.to_xarray()
maintenance_material = dataset_to_array(stacked_maintenance_material_xr, ["Type", "material"], [])


In [ ]:
maintenance_material_pd.columns

In [6]:
import numpy as np
import xarray as xr
from scipy.stats import norm
# Expected value of a Weibull distribution: scale * Γ(1 + 1/shape)
from scipy.special import gamma

def expected_weibull(shape, scale):
    return scale * gamma(1 + 1 / shape)

def expected_folded_normal(mu, sigma):
    return sigma * np.sqrt(2 / np.pi) * np.exp(-mu**2 / (2 * sigma**2)) + mu * (1 - 2 * norm.cdf(-mu / sigma))


def compute_expected_weibull(params):
    shape = params[0]
    scale = params[1]
    return expected_weibull(shape, scale)

def compute_expected_folded(params):
    mu, sigma = params[0], params[1]
    return expected_folded_normal(mu, sigma)

params_folded = prep_data["lifetimes"]["folded_norm"].sel(Time=2020).drop_vars("Time")
params_weibull = prep_data["lifetimes"]["weibull"].sel(Time=2020).drop_vars("Time")

expected_folded = xr.apply_ufunc(
    compute_expected_folded,
    params_folded,
    input_core_dims=[["ScipyParam"]],
    output_core_dims=[[]],
    vectorize=True,
    output_dtypes=[float]
)

expected_weibull = xr.apply_ufunc(
    compute_expected_weibull,
    params_weibull,
    input_core_dims=[["ScipyParam"]],
    output_core_dims=[[]],
    vectorize=True,
    output_dtypes=[float]
)

expected_lifetimes = expected_folded.fillna(expected_weibull)





In [10]:
maintenance_material

<xarray.DataArray (Type: 8, material: 14)> Size: 896B
array([[1.06570556e-01, 8.70654668e-03, 1.47050000e-03, 1.18200000e-03,
        0.00000000e+00, 0.00000000e+00, 1.38000000e-04, 0.00000000e+00,
        2.69986972e-02, 1.35951419e-02, 7.68615910e-02, 1.25310600e-01,
        0.00000000e+00, 0.00000000e+00],
       [1.02067898e-02, 0.00000000e+00, 4.22272727e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        3.23863636e-03, 2.75795455e-03, 0.00000000e+00, 9.79352557e-02,
        0.00000000e+00, 0.00000000e+00],
       [3.50176315e-01, 0.00000000e+00, 6.00302254e-02, 4.62427746e-06,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 2.53971098e-03, 1.50512717e-02, 5.90297217e-01,
        0.00000000e+00, 9.82658960e-07],
       [0.00000000e+00, 0.00000000e+00, 3.00000000e-04, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        1.30000000e-02, 1.20000000e-02, 2.33000000e-01, 2.20000000e-02,
        0.00000000e+00, 0.00000000e+00],
       [1.94650000e-02, 0.00000000e+00, 6.39250000e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        3.75000000e-03, 3.22500000e-03, 0.00000000e+00, 1.86517500e-01,
        0.00000000e+00, 0.00000000e+00],
       [3.82896552e-03, 0.00000000e+00, 1.35827586e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 7.44827586e-04, 0.00000000e+00, 4.35368966e-02,
        0.00000000e+00, 0.00000000e+00],
       [3.22448276e-03, 0.00000000e+00, 1.28844828e-03, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 7.44827586e-04, 0.00000000e+00, 3.81663793e-02,
        0.00000000e+00, 0.00000000e+00],
       [1.21613459e-01, 0.00000000e+00, 2.08480215e-02, 1.12146247e-02,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 4.75797161e-02, 1.58813083e-02, 2.05005545e-01,
        0.00000000e+00, 8.09052282e-03]])
Coordinates:
  * Type      (Type) object 64B 'Cars' 'Heavy Freight Trucks' ... 'Trains'
  * material  (material) object 112B 'Aluminium' 'Co' 'Cu' ... 'Ti' 'Wood'

In [7]:
expected_lifetimes

<xarray.DataArray 'folded_norm' (Type: 16)> Size: 128B
array([ 37.59398496,  74.73309609, 142.85714286,  30.07518797,
       112.78195489, 150.37593985,  97.7443609 ,  71.42857143,
        30.07518797,  97.7443609 ,  40.37267081,  71.17437722,
        40.37267081,  97.7443609 , 131.57894737,  97.7443609 ])
Coordinates:
  * Type     (Type) <U25 2kB 'Bikes' 'Freight Planes' ... 'Very Large Ships'

In [8]:
maintenance_material_per_year = (maintenance_material / expected_lifetimes)
maintenance_material_per_year_broadcasted = vehicle_knowledge_graph.rebroadcast_xarray_impute(
    maintenance_material_per_year, types.values)

prep_data["maintenance_material_fractions"] = maintenance_material_per_year_broadcasted

In [9]:
maintenance_material_per_year_broadcasted

<xarray.DataArray (Type: 53, material: 14)> Size: 6kB
array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
...
       [7.98679576e-05, 0.00000000e+00, 3.19138727e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.84488064e-05, 0.00000000e+00, 9.45351857e-04,
        0.00000000e+00, 0.00000000e+00],
       [7.98679576e-05, 0.00000000e+00, 3.19138727e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.84488064e-05, 0.00000000e+00, 9.45351857e-04,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [9.24262288e-04, 0.00000000e+00, 1.58444964e-04, 8.52311480e-05,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 3.61605843e-04, 1.20697943e-04, 1.55804214e-03,
        0.00000000e+00, 6.14879734e-05],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00]])
Coordinates:
  * Type      (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * material  (material) object 112B 'Aluminium' 'Co' 'Cu' ... 'Ti' 'Wood'

In [ ]:
prep_data["lifetimes"]["weibull"].sel(Time=2020).drop("Time")

In [ ]:
# Sum over all dimensions except Type
sum_by_type = maintenance_material_per_year_broadcasted.sum(dim=["material"])

# Get the Type names where the total sum is 0
types_with_only_zeros = sum_by_type.where(sum_by_type == 0, drop=True).Type.values
types_with_only_zeros


In [ ]:
maintenance_material_per_year_broadcasted

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [ ]:
main_model_normal.simulate(simulation_timeline)

In [ ]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    ).finish()

In [ ]:
#main_model_factory.simulate(simulation_timeline)

In [ ]:
main_model_normal.stock_model.inflow[2020]

In [ ]:
# Sum general inflow over Region
inflow_total = main_model_normal.material_model.inflow_materials[2020].sum(dim="Region")

# Sum maintenance inflow over Region and materials
inflow_maintenance = main_model_normal.maintenance_model.inflow_maintenance[2020].sum(dim=["Region"])#.sel(Type="Trains")

# Convert to pandas Series
inflow_total_pd = inflow_total.to_series()
inflow_maintenance_pd = inflow_maintenance.to_series()

# Combine into a DataFrame
df_plot = pd.DataFrame({
    "Total Inflow": inflow_total_pd,
    "Maintenance Inflow": inflow_maintenance_pd
})


In [ ]:
train_maintenance =  main_model_normal.material_model.inflow_materials[2020].sum(dim="Region"
                        ).sel({"Type": ["Trains", "High Speed Trains"]}).to_series()
train_production = main_model_normal.maintenance_model.inflow_maintenance[2020].sum(dim="Region"
                        ).sel({"Type": ["Trains", "High Speed Trains"]}).to_series()
train_maintenance = train_maintenance.rename_axis(index={"material": "materials"})

# Align and combine into DataFrame
combined = pd.DataFrame({
    "Production": train_production,
    "Maintenance": train_maintenance
}).fillna(0)

# Drop rows where both are zero
df_plot = combined[(combined["Production"] != 0) | (combined["Maintenance"] != 0)]



In [ ]:
train_production =  main_model_normal.material_model.inflow_materials.to_array().sum(dim="Region"
                        ).sel(Type= "High Speed Trains").drop(["Type"])#.sel({"Type": ["Trains", "High Speed Trains"]})
train_maintenance = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim="Region"
                        ).sel(Type= "High Speed Trains").drop(["Type"])#.sel({"Type": ["Trains", "High Speed Trains"]})
train_stock = main_model_normal.material_model.stock_by_cohort_materials.sum(dim="Region"
                        ).sel(Type= "High Speed Trains").drop(["Type"])

train_maintenance

In [ ]:
import matplotlib.pyplot as plt

# Convert to DataFrame
df = train_production.to_pandas()  # shape: (time, material)

# 1. Drop materials that are all zero
df = df.loc[:, (df != 0).any(axis=0)]

# 2. Filter from 1972 onward (assuming time is in datetime or integer years)
df = df[df.index >= 1972]

# Plot
df.plot.area(figsize=(12, 6), colormap='tab20')
plt.xlabel('Time')
plt.ylabel('Material Use')
plt.title('Material Use Over Time (Stacked)')
plt.legend(title='Material', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Step 1: Convert all to DataFrames
maintenance_df = train_maintenance.to_pandas()
production_df = train_production.to_pandas()
stock_df = train_stock.to_pandas()

# Step 2: Filter all materials where at least one of the three is non-zero
nonzero_materials = (
    (maintenance_df != 0).any(axis=0) |
    (production_df != 0).any(axis=0) |
    (stock_df != 0).any(axis=0)
)

# Apply mask to all three
maintenance_df = maintenance_df.loc[:, nonzero_materials]
production_df = production_df.loc[:, nonzero_materials]
stock_df = stock_df.loc[:, nonzero_materials]

# Step 3: Filter from 1972 onward
maintenance_df = maintenance_df[maintenance_df.index >= 1972]
production_df = production_df[production_df.index >= 1972]
stock_df = stock_df[stock_df.index >= 1972]

# Step 4: Plot in 3 side-by-side subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

maintenance_df.plot.area(ax=axes[0], colormap="tab20")
axes[0].set_title("Material Inflow (Maintenance)")
axes[0].set_xlabel("Year")

production_df.plot.area(ax=axes[1], colormap="tab20")
axes[1].set_title("Material Inflow (Production)")
axes[1].set_xlabel("Year")

stock_df.plot.area(ax=axes[2], colormap="tab20")
axes[2].set_title("Material Stock")
axes[2].set_xlabel("Year")

for ax in axes:
    ax.set_ylabel("Mt")
    ax.legend().remove()

# Add one shared legend outside the plot
fig.legend(maintenance_df.columns, title="Material", loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Convert to DataFrames
maintenance_df = train_maintenance.to_pandas()
production_df = train_production.to_pandas()
stock_df = train_stock.to_pandas()

# Filter out zero-only materials
nonzero_materials = (
    (maintenance_df != 0).any(axis=0) |
    (production_df != 0).any(axis=0) |
    (stock_df != 0).any(axis=0)
)

maintenance_df = maintenance_df.loc[:, nonzero_materials]
production_df = production_df.loc[:, nonzero_materials]

# Filter from 1972 onwards
maintenance_df = maintenance_df[maintenance_df.index >= 2000]
production_df = production_df[production_df.index >= 2000]

# Total maintenance sum (used for offset and line)
maintenance_total = maintenance_df.sum(axis=1)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot maintenance materials as stacked area
maintenance_df.plot.area(ax=ax, stacked=True, colormap="tab20", alpha=0.7)
# Plot a thick line on top of maintenance total
ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")

# Plot production materials offset above maintenance
prod_bottom = maintenance_total.copy()
for material in production_df.columns:
    top = prod_bottom + production_df[material]
    ax.fill_between(
        production_df.index,
        prod_bottom,
        top,
        label=f"Prod: {material}",
        alpha=0.7,
        step="mid"
    )
    prod_bottom = top

# Final touches
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Maintenance and Production Material Flows for High-Speed Trains")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()


In [ ]:
train_production =  main_model_normal.material_model.inflow_materials.to_array().sum(dim="Region"
                        ).sel(Type= "Trains").drop(["Type"])#.sel({"Type": ["Trains", "High Speed Trains"]})
train_maintenance = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim="Region"
                        ).sel(Type= "Trains").drop(["Type"])#.sel({"Type": ["Trains", "High Speed Trains"]})
train_stock = main_model_normal.material_model.stock_by_cohort_materials.sum(dim="Region"
                        ).sel(Type= "Trains").drop(["Type"])

train_maintenance

In [ ]:
import matplotlib.pyplot as plt

# Convert to DataFrames
maintenance_df = train_maintenance.to_pandas()
production_df = train_production.to_pandas()
stock_df = train_stock.to_pandas()

# Filter out zero-only materials
nonzero_materials = (
    (maintenance_df != 0).any(axis=0) |
    (production_df != 0).any(axis=0) |
    (stock_df != 0).any(axis=0)
)

maintenance_df = maintenance_df.loc[:, nonzero_materials]
production_df = production_df.loc[:, nonzero_materials]

# Filter from 1972 onwards
maintenance_df = maintenance_df[maintenance_df.index >= 2000]
production_df = production_df[production_df.index >= 2000]

# Total maintenance sum (used for offset and line)
maintenance_total = maintenance_df.sum(axis=1)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Plot maintenance materials as stacked area
maintenance_df.plot.area(ax=ax, stacked=True, colormap="tab20", alpha=0.7)
# Plot a thick line on top of maintenance total
ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")

# Plot production materials offset above maintenance
prod_bottom = maintenance_total.copy()
for material in production_df.columns:
    top = prod_bottom + production_df[material]
    ax.fill_between(
        production_df.index,
        prod_bottom,
        top,
        label=f"Prod: {material}",
        alpha=0.7,
        step="mid"
    )
    prod_bottom = top

# Final touches
ax.set_xlabel("Year")
ax.set_ylabel("Material Flow (Mt)")
ax.set_title("Maintenance and Production Material Flows for Trains")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="Materials")
plt.tight_layout()
plt.show()


In [ ]:
# Sort by total inflow (optional)
df_plot = df_plot.sort_values("Production", ascending=False)

# Plot side-by-side bars with log scale
ax = df_plot.plot(kind="bar", figsize=(12, 6), color=["skyblue", "orange"])
ax.set_yscale("log")
ax.set_ylabel("Inflow [kg]")
ax.set_xlabel("Transport Type")
ax.set_title("Global Material Inflows by Type – 2020")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Define consistent colors
material_colors = {
    "Steel": "#4B4B4B",
    "Aluminium": "#A9A9A9",
    "Others": "#F0E68C",
    "Plastics": "#1E90FF",
    "Copper": "#B87333",
    "Rubber": "#DC143C",
    "Glass": "#00CED1",
    "Wood": "#8B4513",
    "Fluids": "#FF6347",
    "Lead": "#808080",
    "Neodymium": "#D2691E",
    "Cobalt": "#0047AB",
    "Cu": "#B87333"
}

# Helper function to sort columns by total sum
def sort_columns_by_sum(df):
    return df.loc[:, df.sum(axis=0).sort_values(ascending=False).index]

# Helper function for a single train type plot
def plot_material_flow(ax, maint_df, prod_df, title):
    # Filter from 1972
    maint_df = maint_df[maint_df.index >= 2000]
    prod_df = prod_df[prod_df.index >= 2000]

    maint_df = sort_columns_by_sum(maint_df)
    prod_df = sort_columns_by_sum(prod_df)
    
    # Keep only non-zero materials across both
    valid_materials = (
        (maint_df != 0).any(axis=0) |
        (prod_df != 0).any(axis=0)
    )
    maint_df = maint_df.loc[:, valid_materials]
    prod_df = prod_df.loc[:, valid_materials]

    # Consistent material order and colors
    materials = maint_df.columns
    colors = [material_colors.get(mat, "#999999") for mat in materials]

    # Plot maintenance
    maint_df.plot.area(ax=ax, stacked=True, color=colors, alpha=0.85, linewidth=0)
    maintenance_total = maint_df.sum(axis=1)
    ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")

    # Plot production stacked above maintenance
    prod_bottom = maintenance_total.copy()
    for mat in materials:
        top = prod_bottom + prod_df[mat]
        ax.fill_between(
            prod_df.index,
            prod_bottom,
            top,
            label=f"Production: {mat}",
            color=material_colors.get(mat, "#999999"),
            alpha=0.85,
            step="mid"
        )
        prod_bottom = top

    # Styling
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Year", fontsize=12)
    ax.tick_params(labelsize=10)
    ax.grid(visible=True, linestyle="--", alpha=0.3)


# === Load data ===
# Maintenance and production for both types
prod_highspeed = main_model_normal.material_model.inflow_materials.to_array().sum(dim="Region").sel(Type="High Speed Trains").drop("Type").to_pandas()
maint_highspeed = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim="Region").sel(Type="High Speed Trains").drop("Type").to_pandas()

prod_regular = main_model_normal.material_model.inflow_materials.to_array().sum(dim="Region").sel(Type="Trains").drop("Type").to_pandas() 
maint_regular = main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim="Region").sel(Type="Trains").drop("Type").to_pandas()

# === Create side-by-side plot ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_material_flow(axes[0], maint_highspeed, prod_highspeed, "High-Speed Trains")
plot_material_flow(axes[1], maint_regular, prod_regular, "Trains")

# Shared Y label
axes[0].set_ylabel("Material Flow (kg)", fontsize=12)

# Shared legend (outside)
#handles, labels = axes[1].get_legend_handles_labels()
#fig.legend(handles[:len(material_colors)], labels[:len(material_colors)], title="Materials",
#           bbox_to_anchor=(1.02, 0.5), loc="center left", fontsize=10, title_fontsize=11)

# Layout optimization for slides
plt.tight_layout()
plt.subplots_adjust(right=0.85)  # Space for legend
plt.show()


In [ ]:
types_with_maintenance = main_model_normal.maintenance_model.inflow_maintenance.to_array().Type.values

# Sum over all Types for which maintenance exists
prod_all = (
    main_model_normal.material_model.inflow_materials
    .to_array()
    .sel(Type=types_with_maintenance)
    .sum(dim=["Region", "Type"])
    .to_pandas()
)

maint_all = (
    main_model_normal.maintenance_model.inflow_maintenance
    .to_array()
    .sum(dim=["Region", "Type"])
    .to_pandas()
)
maint_all = maint_all[maint_all.index >= 2000]
prod_all = prod_all[prod_all.index >= 2000]

maint_all = sort_columns_by_sum(maint_all)
prod_all = sort_columns_by_sum(prod_all)

# Keep only non-zero materials across both
valid_materials = (
    (maint_all != 0).any(axis=0) |
    (prod_all != 0).any(axis=0)
)
maint_all = maint_all.loc[:, valid_materials]
prod_all = prod_all.loc[:, valid_materials]

# Consistent material order and colors
materials = sort_columns_by_sum(maint_all + prod_all).columns
maint_all = maint_all[materials]
prod_all = prod_all[materials]
colors = [material_colors.get(mat, "#999999") for mat in materials]

# Plot maintenance


fig, ax = plt.subplots(figsize=(8, 6))
maintenance_total = maint_all.sum(axis=1)
#ax.plot(maintenance_total.index, maintenance_total, color="black", linewidth=3, label="Total Maintenance")

maint_all.plot.area(ax=ax, stacked=True, color=colors, alpha=0.85, linewidth=0)



# Plot production stacked above maintenance
prod_bottom = maintenance_total.copy()
for mat in materials:
    #break
    top = prod_bottom + prod_all[mat]
    ax.fill_between(
        prod_all.index,
        prod_bottom,
        top,
        label=f"Production: {mat}",
        color=material_colors.get(mat, "#999999"),
        alpha=0.85,
        step="mid"
    )
    prod_bottom = top

# Styling
ax.set_title("All modes", fontsize=14)
ax.set_xlabel("Year", fontsize=12)
ax.tick_params(labelsize=10)
ax.grid(visible=True, linestyle="--", alpha=0.3)

ax.set_ylabel("Material Flow (kg)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, sharey=True)

# --- Maintenance subplot ---
maint_all.plot.area(ax=ax1, stacked=True, color=colors, alpha=0.85, linewidth=0)
ax1.set_title("Maintenance - All Modes", fontsize=14)
ax1.set_ylabel("Material Flow (kg)", fontsize=12)
ax1.grid(visible=True, linestyle="--", alpha=0.3)
ax1.tick_params(labelsize=10)

# --- Production subplot ---
prod_all.plot.area(ax=ax2, stacked=True, color=colors, alpha=0.85, linewidth=0)
ax2.set_title("Production - All Modes", fontsize=14)
ax2.set_xlabel("Year", fontsize=12)
ax2.set_ylabel("Material Flow (kg)", fontsize=12)
ax2.grid(visible=True, linestyle="--", alpha=0.3)
ax2.tick_params(labelsize=10)

# --- Final layout ---
plt.tight_layout()
plt.show()


In [ ]:
# Total sum across all years and materials
total_prod = prod_all.sum().sum()
total_maint = maint_all.sum().sum()

print(f"Total Production: {total_prod:,.0f}")
print(f"Total Maintenance: {total_maint:,.0f}")
print(f"Maintenance as % of Production: {total_maint / total_prod:.2f}%")


In [ ]:
# Get all Types with maintenance data
types_with_maintenance = main_model_normal.maintenance_model.inflow_maintenance.to_array().Type.values

# Sum over all Types for which maintenance exists
prod_all = (
    main_model_normal.material_model.inflow_materials
    .to_array()
    .sel(Type=types_with_maintenance)
    .sum(dim=["Region", "Type"])
    .to_pandas()
)

maint_all = (
    main_model_normal.maintenance_model.inflow_maintenance
    .to_array()
    .sum(dim=["Region", "Type"])
    .to_pandas()
)

# === Create side-by-side plot ===
fig, ax = plt.subplots(figsize=(8, 6))

plot_material_flow(ax, maint_all, prod_all, "All modes")



ax.set_ylabel("Material Flow (kg)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
maint_all

In [ ]:
# === For High-Speed Trains ===

# Calculate the average additional maintenance relative to production for High-Speed Trains (percentage)
average_additional_relative_to_production_highspeed = (
    maint_highspeed.sum(axis=1).mean(axis=0) / prod_highspeed.sum(axis=1).mean(axis=0)) * 100
# sum over materials, mean over years

# === For Normal Trains ===

# Calculate the average additional maintenance relative to production for Normal Trains (percentage)
average_additional_relative_to_production_regular = (
    maint_regular.sum(axis=1).mean(axis=0).sum() / prod_regular.sum(axis=1).mean(axis=0).sum()) * 100
# sum over materials, mean over years

# Print the results
print(f"Average Additional Maintenance Relative to Production for High-Speed Trains: {average_additional_relative_to_production_highspeed:.2f}%")
print(f"Average Additional Maintenance Relative to Production for Normal Trains: {average_additional_relative_to_production_regular:.2f}%")


In [ ]:
maint_highspeed

In [ ]:
additional_maintenance_regular

In [ ]:
main_model_normal.maintenance_model.inflow_maintenance[2020].sum(dim=["Region"]).sel(Type="Trains")

In [ ]:
main_model_normal.maintenance_model.inflow_maintenance.to_array().sum(dim=["Region"]).sel(Type="Trains")